# Benchmark: Qwen3-2B vs Qwen3-1.7B

Compare the trained 2B model against the 1.7B baseline on:
- **Validation set** (577 examples from `val.jsonl`) — syntax valid, semantic match, exact match, latency
- **Benchmark suite** (301 hand-crafted tests from `benchmark_queries.json`) — pass rate, exact match, per-category breakdown

**1.7B baseline:** 297/301 pass (98.7%), 35/301 exact (11.6%)

**Steps:**
1. Run all cells (takes ~15 min total)
2. Results are saved to `benchmark_results_*.json` in your Colab files
3. Download or copy the summary table at the end

**Requirements:** Colab T4 GPU (free tier works)

## 1. Setup

In [1]:
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv

name, memory.total [MiB], memory.free [MiB]
Tesla T4, 15360 MiB, 14913 MiB


In [2]:
# Install deps — restart runtime after this cell if needed
!pip install -q "transformers>=5.2.0" accelerate peft sentencepiece protobuf tiktoken

In [3]:
# Clone repo for validation modules + data
import os
if not os.path.exists("nls-finetune-scix"):
    !git clone https://github.com/adsabs/nls-finetune-scix.git
else:
    print("Repo already cloned")

# Add to Python path
import sys
sys.path.insert(0, "nls-finetune-scix/packages/finetune/src")

Repo already cloned


In [4]:
# Optional: HuggingFace auth for faster downloads
hf_token = os.environ.get("HF_TOKEN")
# if not hf_token:
#     hf_token = input("Hugging Face token (or press Enter to skip): ").strip()
if hf_token:
    os.environ["HF_TOKEN"] = hf_token
    from huggingface_hub import login
    login(token=hf_token)
    print("Authenticated with Hugging Face")
else:
    print("Skipping auth — downloads may be slower")

Skipping auth — downloads may be slower


## 2. Configuration

Edit the model repos/checkpoints here if they differ.

In [5]:
# ── Model configs ──────────────────────────────────────────
MODELS = {
    "1.7B": {
        "adapter_repo": "adsabs/NLQT-Qwen3-1.7B",
        "checkpoint": "checkpoint-2014",
        "base_model": "Qwen/Qwen3-1.7B",
    },
    "2B": {
        "adapter_repo": "adsabs/NLQT-Qwen3.5-2B",
        "checkpoint": "checkpoint-3891",
        "base_model": "Qwen/Qwen3.5-2B",
    },
}

# ── Data paths ─────────────────────────────────────────────
REPO_DIR = "nls-finetune-scix"
VAL_PATH = f"{REPO_DIR}/data/datasets/processed/val.jsonl"
BENCHMARK_PATH = f"{REPO_DIR}/data/datasets/benchmark/benchmark_queries.json"

# ── Eval settings ─────────────────────────────────────────
VAL_SAMPLE = 100        # Number of val examples (set to None for all 577)
LATENCY_WARMUP = 3      # Warmup queries before timing
MAX_NEW_TOKENS = 256

## 3. Shared Evaluation Functions

In [6]:
import json
import re
import time
from collections import defaultdict
from dataclasses import dataclass, field
from datetime import datetime
from pathlib import Path

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

# Import validation from the repo
from finetune.domains.scix.validate import lint_query, validate_field_constraints
from finetune.domains.scix.constrain import constrain_query_output

SYSTEM_PROMPT = '''Convert natural language to ADS/SciX search query. Output JSON: {"query": "..."}

Example:
User: Query: papers by Hawking on black holes from the 1970s
Date: 2025-03-16
Assistant: {"query": "author:\\"Hawking, S\\" abs:\\"black holes\\" pubdate:[1970 TO 1979]"}'''


def load_model(config: dict) -> tuple:
    """Load a LoRA adapter model. Returns (model, tokenizer)."""
    repo = config["adapter_repo"]
    ckpt = config["checkpoint"]
    base = config["base_model"]
    device = "cuda" if torch.cuda.is_available() else "cpu"

    print(f"Loading tokenizer from {repo}/{ckpt}...")
    tokenizer = AutoTokenizer.from_pretrained(repo, subfolder=ckpt, trust_remote_code=True)

    print(f"Loading base model {base}...")
    base_model = AutoModelForCausalLM.from_pretrained(
        base, torch_dtype=torch.float16, device_map=device, trust_remote_code=True,
    )

    print(f"Applying LoRA adapter from {repo}/{ckpt}...")
    model = PeftModel.from_pretrained(base_model, repo, subfolder=ckpt, torch_dtype=torch.float16)
    model = model.merge_and_unload()
    print(f"Model loaded on {device}")
    return model, tokenizer


def generate(model, tokenizer, nl_text: str, date: str = "2025-12-15") -> tuple[str, float]:
    """Generate ADS query. Returns (query_string, latency_ms)."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Query: {nl_text}\nDate: {date}"},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    prompt_len = inputs["input_ids"].shape[1]

    start = time.time()
    with torch.no_grad():
        out = model.generate(
            **inputs, max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False, pad_token_id=tokenizer.eos_token_id,
        )
    latency_ms = (time.time() - start) * 1000

    response = tokenizer.decode(out[0][prompt_len:], skip_special_tokens=True).strip()

    # Strip <think> block
    if "</think>" in response:
        response = response.split("</think>")[-1].strip()

    # Extract JSON query
    try:
        js = response.find("{")
        je = response.rfind("}") + 1
        if js >= 0 and je > js:
            data = json.loads(response[js:je])
            return data.get("query", response), latency_ms
    except json.JSONDecodeError:
        pass

    return response, latency_ms


def normalize_query(q: str) -> str:
    """Normalize for comparison."""
    q = q.lower()
    q = re.sub(r"\s+", " ", q).strip()
    q = q.replace("\u201c", '"').replace("\u201d", '"')
    return q


def check_semantic_match(expected: str, output: str) -> bool:
    """Check >=50% field:value overlap."""
    def extract(q):
        comps = set()
        for m in re.finditer(r"(\w+):(?:\"([^\"]+)\"|(\S+))", q, re.I):
            comps.add(f"{m.group(1).lower()}:{(m.group(2) or m.group(3)).lower()}")
        for m in re.finditer(r"(citations|references|trending|similar|useful|reviews|topn)\s*\(", q, re.I):
            comps.add(f"op:{m.group(1).lower()}")
        return comps

    exp_c = extract(expected)
    out_c = extract(output)
    if not exp_c:
        return True
    return len(exp_c & out_c) / len(exp_c) >= 0.5


print("Evaluation functions loaded")

Evaluation functions loaded


## 4. Load Validation & Benchmark Data

In [7]:
# Load validation set
val_examples = []
with open(VAL_PATH) as f:
    for line in f:
        data = json.loads(line)
        msgs = data["messages"]
        nl = expected = None
        for m in msgs:
            if m["role"] == "user":
                content = m["content"]
                if "Query:" in content:
                    nl = content.split("Query:")[1].split("\n")[0].strip()
            elif m["role"] == "assistant":
                try:
                    expected = json.loads(m["content"]).get("query", "")
                except json.JSONDecodeError:
                    expected = m["content"]
        if nl and expected:
            val_examples.append({"nl": nl, "expected": expected})

if VAL_SAMPLE and VAL_SAMPLE < len(val_examples):
    val_examples = val_examples[:VAL_SAMPLE]

print(f"Validation examples: {len(val_examples)}")

# Load benchmark suite
with open(BENCHMARK_PATH) as f:
    benchmark = json.load(f)

# Flatten benchmark into list of (test_dict, category, subcategory)
bench_tests = []
for section in ["field_types", "operators", "enum_fields"]:
    if section in benchmark:
        for subcat, tests in benchmark[section].items():
            for t in tests:
                bench_tests.append((t, section, subcat))
for section in ["edge_cases", "regression_tests"]:
    if section in benchmark:
        for t in benchmark[section]:
            bench_tests.append((t, section, section.rstrip("s")))

print(f"Benchmark tests: {len(bench_tests)}")

Validation examples: 100
Benchmark tests: 301


## 5. Run Evaluation

Evaluates each model sequentially. The 1.7B model is unloaded before loading the 2B to stay within T4 VRAM.

In [8]:
import gc


def extract_operators(query: str) -> set[str]:
    return set(m.group(1) for m in re.finditer(
        r"\b(citations|references|trending|similar|useful|reviews|topn)\s*\(", query, re.I
    ))


def eval_benchmark_test(model, tokenizer, test: dict, category: str, subcategory: str) -> dict:
    """Evaluate one benchmark test case. Returns result dict."""
    nl = test.get("natural_language", "")
    expected_query = test.get("expected_query")
    expected_behavior = test.get("expected_behavior")
    forbidden = test.get("forbidden_patterns", [])
    test_id = test.get("id", "unknown")
    difficulty = test.get("difficulty", "simple")

    try:
        actual, latency = generate(model, tokenizer, nl)
        # Apply constraint filter (same as production pipeline)
        actual = constrain_query_output(actual)
    except Exception as e:
        actual, latency = f"ERROR: {e}", 0

    lint = lint_query(actual)
    syntax_valid = lint.valid
    constraint = validate_field_constraints(actual)
    constraint_valid = constraint.valid
    forbidden_matched = [p for p in forbidden if p.lower() in actual.lower()]

    exact_match = False
    if expected_query:
        exact_match = normalize_query(expected_query) == normalize_query(actual)

    # Determine pass/fail (same logic as evaluate_benchmark.py)
    passed = False
    if forbidden:
        passed = len(forbidden_matched) == 0 and syntax_valid
    elif expected_behavior:
        if expected_behavior == "no_operator":
            passed = not bool(extract_operators(actual)) and syntax_valid
        elif expected_behavior == "single_operator":
            passed = len(extract_operators(actual)) <= 1 and syntax_valid
        elif expected_behavior == "valid_response":
            passed = syntax_valid or not actual.startswith("ERROR")
        elif expected_behavior == "passthrough":
            passed = syntax_valid
        elif expected_behavior == "balanced_parens":
            passed = actual.count("(") == actual.count(")") and syntax_valid
        elif expected_behavior in ("ambiguous", "valid_database_only", "valid_property_only"):
            passed = constraint_valid and syntax_valid
        else:
            passed = syntax_valid
    elif expected_query:
        passed = syntax_valid and constraint_valid and len(forbidden_matched) == 0

    return {
        "test_id": test_id, "category": category, "subcategory": subcategory,
        "difficulty": difficulty, "nl": nl, "expected": expected_query,
        "actual": actual, "exact_match": exact_match, "syntax_valid": syntax_valid,
        "constraint_valid": constraint_valid, "passed": passed, "latency_ms": latency,
        "forbidden_matched": forbidden_matched,
    }


def run_full_eval(model_name: str, config: dict) -> dict:
    """Run full evaluation for one model. Returns results dict."""
    print(f"\n{'='*60}")
    print(f"Evaluating: {model_name} ({config['adapter_repo']})")
    print(f"{'='*60}")

    model, tokenizer = load_model(config)

    # ── Warmup ──
    print(f"\nWarming up ({LATENCY_WARMUP} queries)...")
    for _ in range(LATENCY_WARMUP):
        generate(model, tokenizer, "test warmup query")

    # ── Validation set ──
    print(f"\nRunning validation set ({len(val_examples)} examples)...")
    val_results = []
    for i, ex in enumerate(val_examples):
        output, latency = generate(model, tokenizer, ex["nl"])
        output = constrain_query_output(output)
        syntax_ok = lint_query(output).valid
        sem_match = check_semantic_match(ex["expected"], output)
        exact = normalize_query(ex["expected"]) == normalize_query(output)
        val_results.append({
            "nl": ex["nl"], "expected": ex["expected"], "output": output,
            "syntax_valid": syntax_ok, "semantic_match": sem_match,
            "exact_match": exact, "latency_ms": latency,
        })
        if (i + 1) % 25 == 0:
            print(f"  {i+1}/{len(val_examples)}...")

    # ── Benchmark suite ──
    print(f"\nRunning benchmark suite ({len(bench_tests)} tests)...")
    bench_results = []
    for i, (test, cat, subcat) in enumerate(bench_tests):
        result = eval_benchmark_test(model, tokenizer, test, cat, subcat)
        bench_results.append(result)
        if (i + 1) % 50 == 0:
            print(f"  {i+1}/{len(bench_tests)}...")

    # ── Compute stats ──
    n_val = len(val_results)
    val_syntax = sum(1 for r in val_results if r["syntax_valid"])
    val_semantic = sum(1 for r in val_results if r["semantic_match"])
    val_exact = sum(1 for r in val_results if r["exact_match"])
    val_latencies = sorted(r["latency_ms"] for r in val_results)

    n_bench = len(bench_results)
    bench_passed = sum(1 for r in bench_results if r["passed"])
    bench_exact = sum(1 for r in bench_results if r["exact_match"])
    bench_syntax = sum(1 for r in bench_results if r["syntax_valid"])

    # Per-category stats
    cat_stats = defaultdict(lambda: {"total": 0, "passed": 0, "exact": 0})
    for r in bench_results:
        cat_stats[r["category"]]["total"] += 1
        cat_stats[r["category"]]["passed"] += r["passed"]
        cat_stats[r["category"]]["exact"] += r["exact_match"]

    # Latency percentiles
    def pct(arr, p):
        return arr[int(len(arr) * p)] if arr else 0

    summary = {
        "model": model_name,
        "adapter_repo": config["adapter_repo"],
        "validation": {
            "total": n_val,
            "syntax_valid": val_syntax,
            "syntax_valid_pct": round(val_syntax / n_val * 100, 1),
            "semantic_match": val_semantic,
            "semantic_match_pct": round(val_semantic / n_val * 100, 1),
            "exact_match": val_exact,
            "exact_match_pct": round(val_exact / n_val * 100, 1),
            "latency_p50_ms": round(pct(val_latencies, 0.5), 1),
            "latency_p95_ms": round(pct(val_latencies, 0.95), 1),
            "latency_p99_ms": round(pct(val_latencies, 0.99), 1),
            "latency_max_ms": round(val_latencies[-1], 1) if val_latencies else 0,
        },
        "benchmark": {
            "total": n_bench,
            "passed": bench_passed,
            "pass_rate_pct": round(bench_passed / n_bench * 100, 1),
            "exact_match": bench_exact,
            "exact_match_pct": round(bench_exact / n_bench * 100, 1),
            "syntax_valid": bench_syntax,
            "syntax_valid_pct": round(bench_syntax / n_bench * 100, 1),
            "by_category": {k: v for k, v in sorted(cat_stats.items())},
        },
        "failed_tests": [r for r in bench_results if not r["passed"]],
        "val_results": val_results,
    }

    # Print summary
    print(f"\n--- {model_name} Results ---")
    print(f"  Val: syntax={val_syntax}/{n_val} ({summary['validation']['syntax_valid_pct']}%)  "
          f"semantic={val_semantic}/{n_val} ({summary['validation']['semantic_match_pct']}%)  "
          f"exact={val_exact}/{n_val} ({summary['validation']['exact_match_pct']}%)")
    print(f"  Val latency: p50={summary['validation']['latency_p50_ms']}ms  "
          f"p95={summary['validation']['latency_p95_ms']}ms  "
          f"p99={summary['validation']['latency_p99_ms']}ms")
    print(f"  Bench: pass={bench_passed}/{n_bench} ({summary['benchmark']['pass_rate_pct']}%)  "
          f"exact={bench_exact}/{n_bench} ({summary['benchmark']['exact_match_pct']}%)")
    for cat, s in sorted(cat_stats.items()):
        print(f"    {cat:20s} {s['passed']}/{s['total']} pass  {s['exact']} exact")

    # Unload model to free VRAM
    del model, tokenizer
    gc.collect()
    torch.cuda.empty_cache()
    print(f"\nModel unloaded, VRAM freed.")

    return summary

In [9]:
# Run both models
all_results = {}
for name, config in MODELS.items():
    all_results[name] = run_full_eval(name, config)


Evaluating: 1.7B (adsabs/NLQT-Qwen3-1.7B)
Loading tokenizer from adsabs/NLQT-Qwen3-1.7B/checkpoint-2014...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/375 [00:00<?, ?B/s]

checkpoint-2014/tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

...

/usr/local/lib/python3.12/dist-packages/peft/peft_model.py:598: UserWarning: Found missing adapter keys while loading the checkpoint: ['base_model.model.model.layers.0.mlp.gate_proj.lora_A.default.weight', 'base_model.model.model.layers.0.mlp.gate_proj.lora_B.default.weight', 'base_model.model.model.layers.0.mlp.up_proj.lora_A.default.weight', 'base_model.model.model.layers.0.mlp.up_proj.lora_B.default.weight', 'base_model.model.model.layers.0.mlp.down_proj.lora_A.default.weight', 'base_model.model.model.layers.0.mlp.down_proj.lora_B.default.weight', 'base_model.model.model.layers.1.mlp.gate_proj.lora_A.default.weight', 'base_model.model.model.layers.1.mlp.gate_proj.lora_B.default.weight', 'base_model.model.model.layers.1.mlp.up_proj.lora_A.default.weight', 'base_model.model.model.layers.1.mlp.up_proj.lora_B.default.weight', 'base_model.model.model.layers.1.mlp.down_proj.lora_A.default.weight', 'base_model.model.model.layers.1.mlp.down_proj.lora_B.default.weight', 'base_model.model.mod

Model loaded on cuda

Warming up (3 queries)...

Running validation set (100 examples)...
  25/100...
  50/100...
  75/100...


  100/100...

Running benchmark suite (301 tests)...
  50/301...
  100/301...
  150/301...
  200/301...
  250/301...


  300/301...



--- 2B Results ---
  Val: syntax=49/100 (49.0%)  semantic=16/100 (16.0%)  exact=3/100 (3.0%)
  Val latency: p50=1676.2ms  p95=2740.7ms  p99=3226.8ms
  Bench: pass=80/301 (26.6%)  exact=6/301 (2.0%)
    edge_cases           11/45 pass  0 exact
    enum_fields          11/55 pass  0 exact
    field_types          33/66 pass  6 exact
    operators            15/105 pass  0 exact
    regression_tests     10/30 pass  0 exact

Model unloaded, VRAM freed.


## 6. Comparison Table

In [10]:
# Side-by-side comparison
print("\n" + "=" * 70)
print("COMPARISON: 1.7B vs 2B")
print("=" * 70)

header = f"{'Metric':<35} {'1.7B':>12} {'2B':>12} {'Delta':>10}"
print(f"\n{header}")
print("-" * 70)

def row(label, key_path, fmt=".1f", suffix="%"):
    v1 = all_results["1.7B"]
    v2 = all_results["2B"]
    for k in key_path:
        v1 = v1[k]
        v2 = v2[k]
    delta = v2 - v1
    sign = "+" if delta > 0 else ""
    print(f"{label:<35} {v1:>11{fmt}}{suffix} {v2:>11{fmt}}{suffix} {sign}{delta:>8{fmt}}{suffix}")

def row_int(label, key_path):
    v1 = all_results["1.7B"]
    v2 = all_results["2B"]
    for k in key_path:
        v1 = v1[k]
        v2 = v2[k]
    delta = v2 - v1
    sign = "+" if delta > 0 else ""
    print(f"{label:<35} {v1:>12} {v2:>12} {sign}{delta:>10}")

print("\nValidation Set:")
row("  Syntax Valid", ["validation", "syntax_valid_pct"])
row("  Semantic Match", ["validation", "semantic_match_pct"])
row("  Exact Match", ["validation", "exact_match_pct"])
row("  Latency p50", ["validation", "latency_p50_ms"], fmt=".0f", suffix="ms")
row("  Latency p95", ["validation", "latency_p95_ms"], fmt=".0f", suffix="ms")

print("\nBenchmark Suite (301 tests):")
row("  Pass Rate", ["benchmark", "pass_rate_pct"])
row("  Exact Match", ["benchmark", "exact_match_pct"])
row("  Syntax Valid", ["benchmark", "syntax_valid_pct"])

# Per-category benchmark
print("\nBenchmark by Category:")
cats = set()
for r in all_results.values():
    cats.update(r["benchmark"]["by_category"].keys())
for cat in sorted(cats):
    s1 = all_results["1.7B"]["benchmark"]["by_category"].get(cat, {"passed": 0, "total": 0, "exact": 0})
    s2 = all_results["2B"]["benchmark"]["by_category"].get(cat, {"passed": 0, "total": 0, "exact": 0})
    pct1 = s1["passed"] / s1["total"] * 100 if s1["total"] else 0
    pct2 = s2["passed"] / s2["total"] * 100 if s2["total"] else 0
    delta = pct2 - pct1
    sign = "+" if delta > 0 else ""
    print(f"  {cat:<33} {s1['passed']}/{s1['total']:>3} ({pct1:>5.1f}%)  "
          f"{s2['passed']}/{s2['total']:>3} ({pct2:>5.1f}%)  {sign}{delta:>5.1f}%")

print()


COMPARISON: 1.7B vs 2B

Metric                                      1.7B           2B      Delta
----------------------------------------------------------------------

Validation Set:
  Syntax Valid                            100.0%        49.0%    -51.0%
  Semantic Match                           72.0%        16.0%    -56.0%
  Exact Match                              25.0%         3.0%    -22.0%
  Latency p50                              1156ms        1676ms +     520ms
  Latency p95                              1844ms        2741ms +     897ms

Benchmark Suite (301 tests):
  Pass Rate                                93.7%        26.6%    -67.1%
  Exact Match                              17.6%         2.0%    -15.6%
  Syntax Valid                             96.7%        26.2%    -70.5%

Benchmark by Category:
  edge_cases                        36/ 45 ( 80.0%)  11/ 45 ( 24.4%)  -55.6%
  enum_fields                       54/ 55 ( 98.2%)  11/ 55 ( 20.0%)  -78.2%
  field_types         

## 7. Failure Analysis

In [11]:
# Show failures unique to each model
for name in ["1.7B", "2B"]:
    failures = all_results[name].get("failed_tests", [])
    print(f"\n--- {name}: {len(failures)} failed benchmark tests ---")
    for f in failures[:15]:
        print(f"  [{f['test_id']}] {f['category']}/{f['subcategory']}")
        print(f"    NL: {f['nl'][:70]}...")
        if f.get("expected"):
            print(f"    Expected: {f['expected'][:60]}")
        print(f"    Got:      {f['actual'][:60]}")
        if f.get("forbidden_matched"):
            print(f"    Forbidden: {f['forbidden_matched']}")
        print()
    if len(failures) > 15:
        print(f"  ... and {len(failures) - 15} more")

# Show tests that differ between models
f1_ids = {f["test_id"] for f in all_results["1.7B"].get("failed_tests", [])}
f2_ids = {f["test_id"] for f in all_results["2B"].get("failed_tests", [])}
print(f"\nFailed by 1.7B only (2B fixed): {f1_ids - f2_ids or 'none'}")
print(f"Failed by 2B only (1.7B passed): {f2_ids - f1_ids or 'none'}")
print(f"Failed by both:                  {f1_ids & f2_ids or 'none'}")


--- 1.7B: 19 failed benchmark tests ---
  [content-003] field_types/content
    NL: papers mentioning exoplanets in the abstract...
    Expected: abstract:"exoplanets"
    Got:      mention_count:[1 TO *] abs:"exoplanets"

  [author-011] field_types/author
    NL: papers by Penrose on cosmology from Cambridge...
    Expected: author:Penrose abs:"cosmology" aff:Cambridge
    Got:      {"query": "author:\"Penrose, R\" abs:\"cosmology\" OR abs:\"

  [date-006] field_types/date
    NL: papers published in January 2023...
    Expected: pubdate:[2023-01 TO 2023-01]
    Got:      pubdate:[2023-01] doctype:article

  [doctype-002] enum_fields/doctype
    NL: PhD theses on exoplanets...
    Expected: doctype:phdthesis abs:"exoplanets"
    Got:      thesisabs:"exoplanets"

  [edge-001] edge_cases/edge_case
    NL: citing behavior of astronomers...
    Got:      citations(abs:"behavior of astronomers")

  [edge-004] edge_cases/edge_case
    NL: similar wavelengths in spectroscopy...
    Got:    

## 8. Save Results

In [12]:
# Save full results
timestamp = datetime.now().strftime("%Y%m%d-%H%M%S")

# Comparison summary (compact)
comparison = {
    "timestamp": datetime.now().isoformat(),
    "models": {name: {"adapter_repo": r["adapter_repo"]} for name, r in all_results.items()},
    "validation_sample": VAL_SAMPLE or len(val_examples),
    "benchmark_total": len(bench_tests),
    "comparison": {},
}
for name, r in all_results.items():
    comparison["comparison"][name] = {
        "validation": r["validation"],
        "benchmark": {k: v for k, v in r["benchmark"].items() if k != "by_category"},
        "benchmark_by_category": r["benchmark"]["by_category"],
    }

out_path = f"benchmark_comparison_{timestamp}.json"
with open(out_path, "w") as f:
    json.dump(comparison, f, indent=2)
print(f"Comparison saved to: {out_path}")

# Full results per model (with all val results + failed tests)
for name, r in all_results.items():
    path = f"benchmark_{name}_{timestamp}.json"
    with open(path, "w") as f:
        json.dump(r, f, indent=2, default=str)
    print(f"{name} full results saved to: {path}")

Comparison saved to: benchmark_comparison_20260318-030710.json
1.7B full results saved to: benchmark_1.7B_20260318-030710.json
2B full results saved to: benchmark_2B_20260318-030710.json


## 9. Decision

Based on the results above:

| Outcome | Action |
|---------|--------|
| 2B significantly better (>3% gain on benchmark pass rate or >5% on exact match) | Adopt 2B as default model |
| 2B similar or marginally better | Stay with 1.7B (lower latency, same quality) |
| 2B worse | Stay with 1.7B, investigate data quality (Workstream F) |
| Latency > 1.5s p95 | 2B too slow for production even if better quality |